In [26]:
!python --version

Python 3.11.9


In [27]:
%matplotlib inline

In [28]:
import os

def define_auth():
    os.environ["HF_TOKEN"] = ""

In [29]:
import pandas as pd

def get_dataset(type="train"):
    splits = {'train': 'train_meta.csv', 'test': 'test_meta.csv'}
    df = pd.read_csv("hf://datasets/lasfk/EEG-fNIRS-based-Handwriting-Trajectory-Dataset/" + splits[type])
    return df

## EEG Extraction

In [30]:
import mne

def read_bdf(subject, session):
    path = f"datasets/raw/{subject}/EEG/{session}.bdf"
    raw = mne.io.read_raw_bdf(path, preload=True)
    return raw

In [31]:
import numpy as np

def clean_bdf(raw):
    empty_channels = [
    'Fpz',
    'Fp1',
    'Fp2',
    'AF3',
    'AF4',
    'AF7',
    'AF8',
    'F7',
    'F8',
    'FT7',
    'FT8',
    'T7',
    'T8',
    'TP7',
    'TP8',
    'Pz',
    'P3',
    'P4',
    'P5',
    'P6',
    'P7',
    'P8',
    'POz',
    'PO3',
    'PO4',
    'PO5',
    'PO6',
    'PO7',
    'PO8',
    'Oz',
    'O1',
    'O2',
    'ECG',
    'HEOR',
    'HEOL',
    'VEOU',
    'VEOL',
    ]
    raw.drop_channels(empty_channels)

def segmentation_eeg(raw, onset_sec, label, tmin, tmax):
    sfreq = raw.info["sfreq"]
    start_sample = int((onset_sec + tmin) * sfreq)
    end_sample = int((onset_sec + tmax) * sfreq)
    
    segment = raw.get_data(
        start=start_sample,
        stop=end_sample
    )

    expected_length = int((tmax - tmin) * sfreq)

    # Skip broken segment
    if segment.shape[1] != expected_length:
        return [], -1

    return segment, label

def normalize_channel(X):
    X_norm = np.zeros_like(X)

    for i in range(X.shape[0]):
    
        for ch in range(X.shape[1]):
    
            signal = X[i, ch]
    
            mean = signal.mean()
            std = signal.std()
    
            if std == 0:
                std = 1e-8
    
            X_norm[i, ch] = (signal - mean) / std
    return X_norm

def preprocessing_bdf(raw):
    clean_bdf(raw)

    # Buang sinyal listrik
    raw.notch_filter(50)

    # Band Pass 0.5Hz~40Hz, sinyal otak kecil, biasanya dalam range tersebut
    raw.filter(0.5, 40)

    # average reference
    raw.set_eeg_reference("average")

    return raw

## fNIRS Extraction

In [32]:
import pandas as pd

def read_fnirs(subject, session):
    path = f"datasets/raw/{subject}/fNIRS/{session}.csv"
    fnirs = pd.read_csv(path)
    return fnirs

In [33]:
import numpy as np
import pandas as pd

from scipy.signal import butter
from scipy.signal import sosfiltfilt
from tqdm import tqdm

def bandpass_filter(data, low, high, fs, order=4):
    nyquist = 0.5 * fs

    low = low / nyquist
    high = high / nyquist

    sos = butter(
        order,
        [low, high],
        btype="band",
        output="sos"
    )

    filtered = sosfiltfilt(
        sos,
        data
    )
    return filtered

def clean_fnirs(df, fs, fnirs_low, fnirs_high):

    fnirs_cols = [
        col for col in df.columns
        if col.startswith("fnirs_")
    ]

    # remove NaN
    df = df.dropna().reset_index(drop=True)

    # filtering
    for col in fnirs_cols:
        signal = df[col].values
        filtered = bandpass_filter(
            signal,
            fnirs_low,
            fnirs_high,
            fs
        )
        df[col] = filtered
    return df

def segmentation_fnirs(
    df,
    onset_sec,
    label,
    tmin,
    tmax,
    fs
):
    fnirs_cols = [
        col for col in df.columns
        if col.startswith("fnirs_")
    ]
    start = int((onset_sec + tmin) * fs)
    end = int((onset_sec + tmax) * fs)

    segment = df.iloc[start:end]
    expected_len = int((tmax - tmin) * fs)

    # skip broken segment
    if len(segment) != expected_len:
        return None, -1

    signal = segment[fnirs_cols].values.T

    return signal, label

def normalize_fnirs(X):
    X_norm = np.zeros_like(X)
    for i in range(X.shape[0]):

        for ch in range(X.shape[1]):

            signal = X[i, ch]

            mean = signal.mean()
            std = signal.std()

            if std == 0:
                std = 1e-8

            X_norm[i, ch] = (
                signal - mean
            ) / std

    return X_norm
    
def preprocessing_fnirs(df, fs, fnirs_low, fnirs_high):
    return clean_fnirs(df, fs, fnirs_low, fnirs_high)

## Extract Features

In [34]:
define_auth()
df_train = get_dataset(type="train")

print(df_train.head())

      trial_id subject  session  onset_sec  event_code  label
0  sub_01_1_00  sub_01        1     25.060         203      3
1  sub_01_1_01  sub_01        1     49.410         201      1
2  sub_01_1_02  sub_01        1     73.870         201      1
3  sub_01_1_03  sub_01        1     98.695         201      1
4  sub_01_1_04  sub_01        1    123.850         202      2


### Perbedaan trial onset_sec diff time

In [35]:
df_train["next_onset"] = (
    df_train["onset_sec"]
    .shift(-1)
)

df_train["diff"] = (
    df_train["next_onset"]
    -
    df_train["onset_sec"]
)

print(df_train["diff"].describe())

count    6442.000000
mean        0.137596
std       151.903929
min     -1345.378000
25%        24.852000
50%        24.858000
75%        24.864000
max       366.888000
Name: diff, dtype: float64


In [36]:
def clean_multimodal(
    X_eeg,
    X_fnirs,
    y,
    subject_ids,
):

    clean_X_eeg = []
    clean_X_fnirs = []
    clean_y = []
    clean_subject_ids = []

    # 150 µV
    THRESHOLD = 150e-6

    for i in range(len(X_eeg)):

        eeg_signal = X_eeg[i]

        # cek apakah noisy
        if np.max(np.abs(eeg_signal)) < THRESHOLD:

            clean_X_eeg.append(
                X_eeg[i]
            )

            clean_X_fnirs.append(
                X_fnirs[i]
            )

            clean_y.append(
                y[i]
            )

            clean_subject_ids.append(
                subject_ids[i]
            )

    clean_X_eeg = np.array(
        clean_X_eeg
    )

    clean_X_fnirs = np.array(
        clean_X_fnirs
    )

    clean_y = np.array(
        clean_y
    )
    clean_subject_ids = np.array(
        clean_subject_ids
    )

    return (
        clean_X_eeg,
        clean_X_fnirs,
        clean_y,
        clean_subject_ids,
    )

### Extract Features EEG & fNIRS

In [37]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

mne.set_log_level("ERROR")

last_subject=""
last_session=0
raw=None
df_fnirs=None

EEG_TMIN = 0.0
EEG_TMAX = 4.0

# Dari explanatory data analysis
FNIRS_FS = 64

# 0.01 Hz ≤ f ≤ 0.2 Hz
# Kita hanya ambil perubahan sinyal yang:
# tidak terlalu lambat
# tidak terlalu cepat
FNIRS_LOW = 0.01
FNIRS_HIGH = 0.2

# Dimulai + 2 Second = menunggu respons aliran darah mulai muncul.
FNIRS_TMIN = 2.0
FNIRS_TMAX = 8.0

X_eeg = []
X_fnirs = []
y = []
subject_ids = []
for index, row in  tqdm(df_train.iterrows(), total=len(df_train)):
    subject = row["subject"]
    session = row["session"]
    onset_sec = row["onset_sec"]
    label = row["label"]

    if last_subject != subject or last_session != session:
        # EEG
        raw = read_bdf(subject, session)
        raw = preprocessing_bdf(raw)

        # fNIRS
        df_fnirs = read_fnirs(subject, session)
        df_fnirs = preprocessing_fnirs(df_fnirs, FNIRS_FS, FNIRS_LOW, FNIRS_HIGH)
        
        last_subject = subject
        last_session = session

    # Segmentation BDF
    segment_eeg, label_eeg = segmentation_eeg(raw, onset_sec, label, EEG_TMIN, EEG_TMAX)
    
    # Segmentation fNIRS
    segment_fnirs, label_fnirs = segmentation_fnirs(
        df_fnirs,
        onset_sec,
        label,
        FNIRS_TMIN,
        FNIRS_TMAX,
        FNIRS_FS
    )
    if label_eeg == -1 or label_fnirs == -1:
        continue

    subject_ids.append(subject)
    X_eeg.append(segment_eeg)
    X_fnirs.append(segment_fnirs)
    y.append(label)

subject_ids = np.array(subject_ids)
X_eeg = np.array(X_eeg)
X_fnirs = np.array(X_fnirs)
y = np.array(y)

# Cleaning multimodal
X_eeg, X_fnirs, y, subject_ids = clean_multimodal(
    X_eeg,
    X_fnirs,
    y,
    subject_ids,
)

X_eeg = np.nan_to_num(
    X_eeg,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X_fnirs = np.nan_to_num(
    X_fnirs,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

# Normalize EEG
X_eeg = normalize_channel(X_eeg)

# Normalize fNIRS
X_fnirs = normalize_fnirs(X_fnirs)

    


100%|██████████████████████████████████████████████████████████████████████████████| 6443/6443 [06:39<00:00, 16.14it/s]


## Training Model

### Check Device

In [38]:
import torch

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(DEVICE)

True
NVIDIA GeForce RTX 4070
cuda


## Classifier

### Init Dataset Class

In [39]:
import torch
from torch.utils.data import Dataset

class MultiModalDataset(Dataset):
    def __init__(self, eeg, fnirs, labels):
        self.eeg = eeg
        self.fnirs = fnirs
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        eeg = torch.tensor(self.eeg[idx], dtype=torch.float32)
        fnirs = torch.tensor(self.fnirs[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return eeg, fnirs, label


### Init SE Block

SE Block = Squeeze-and-Excitation Block, berfungsi untuk memberikan attention ke channel penting 

In [40]:
import torch.nn as nn

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels),
            nn.ReLU(),
            nn.Linear(channels, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y

### Encoder untuk EEG

In [41]:
def split_eeg_windows(eeg_signal):

    early = eeg_signal[:, :, 0:250]
    mid = eeg_signal[:, :, 250:500]
    late = eeg_signal[:, :, 500:750]

    return early, mid, late

In [42]:
import torch.nn as nn

class EEGEncoder(nn.Module):
    def __init__(self, eeg_channels):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(
                eeg_channels,
                64,
                kernel_size=7,
                padding=3
            ),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(
                64,
                128,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(128),
            nn.GELU(),
            SEBlock(128),
            nn.MaxPool1d(2),
            nn.Conv1d(
                128,
                256,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()
        )

        self.proj = nn.Sequential(
            nn.Linear(
                256,
                EMBED_DIM
            ),
            nn.LayerNorm(
                EMBED_DIM
            ),
            nn.Dropout(0.3)
        )

    def forward(self, x):
        x = self.net(x)
        x = self.proj(x)
        return x

### Encoder untuk fNIRS 

In [43]:
import torch.nn as nn

class FNIRSEncoder(nn.Module):
    def __init__(self, fnirs_channels):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(
                fnirs_channels,
                32,
                kernel_size=11,
                padding=5
            ),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(
                32,
                64,
                kernel_size=9,
                padding=4
            ),
            nn.BatchNorm1d(64),
            nn.GELU(),
            SEBlock(64),
            nn.MaxPool1d(2),
            nn.Conv1d(
                64,
                128,
                kernel_size=7,
                padding=3
            ),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()
        )

        self.proj = nn.Sequential(
            nn.Linear(128, EMBED_DIM),
            nn.LayerNorm(EMBED_DIM),
            nn.Dropout(0.3)
        )

    def forward(self, x):
        x = self.net(x)
        x = self.proj(x)
        return x

### Fusion EEG + fNIRS

In [44]:
import torch.nn as nn

class GatedFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )

    def forward(self, eeg_feat, fnirs_feat):
        fusion_input = torch.cat([eeg_feat, fnirs_feat], dim=1)
        gate = self.gate(fusion_input)

        fused = (gate * eeg_feat + (1 - gate) * fnirs_feat)
        return fused

### Init MultiModalNet Class

In [45]:
import torch
import torch.nn as nn

class MultiModalNet(nn.Module):
    def __init__(self, eeg_channels, fnirs_channels, num_classes):
        super().__init__()

        self.eeg_encoder = EEGEncoder(eeg_channels)
        self.fnirs_encoder = FNIRSEncoder(fnirs_channels)
        self.fusion = GatedFusion(EMBED_DIM)
        self.classifier = nn.Sequential(
            nn.Linear(
                EMBED_DIM,
                128
            ),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(
                128,
                64
            ),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(
                64,
                num_classes
            )
        )

    def forward(self, eeg, fnirs):
        # EEG Temporal windows
        early, mid, late = split_eeg_windows(
            eeg
        )

        early_feat = self.eeg_encoder(
            early
        )

        mid_feat = self.eeg_encoder(
            mid
        )

        late_feat = self.eeg_encoder(
            late
        )

        eeg_feat = torch.cat(
            [
                early_feat,
                mid_feat,
                late_feat
            ],
            dim=1
        )

        eeg_feat = eeg_feat.view(
            eeg_feat.shape[0],
            3,
            EMBED_DIM
        ).mean(dim=1)

        # fNIRS delay compensation
        fnirs = fnirs[
            :,
            :,
            FNIRS_DELAY_START:FNIRS_DELAY_END
        ]

        fnirs_feat = self.fnirs_encoder(
            fnirs
        )


        # Modality dropout
        if self.training:

            if torch.rand(1).item() < 0.15:
                eeg_feat = torch.zeros_like(
                    eeg_feat
                )

            if torch.rand(1).item() < 0.15:
                fnirs_feat = torch.zeros_like(
                    fnirs_feat
                )

        # Gated Fusion
        fused = self.fusion(
            eeg_feat,
            fnirs_feat
        )

        out = self.classifier(
            fused
        )

        return out

### Wrap function untuk training 1 EPOCH

In [47]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
)

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler
):
    model.train()
    total_loss = 0

    preds_all = []
    labels_all = []

    for eeg, fnirs, labels in loader:
        eeg = eeg.to(DEVICE)
        fnirs = fnirs.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        
        with torch.amp.autocast("cuda"):
            outputs = model(eeg, fnirs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        preds_all.extend(
            preds.cpu().numpy()
        )
        labels_all.extend(
            labels.cpu().numpy()
        )

    acc = accuracy_score(
        labels_all,
        preds_all
    )
    bal_acc = balanced_accuracy_score(
        labels_all,
        preds_all
    )
    return (
        total_loss / len(loader),
        acc,
        bal_acc
    )

### Validate Function

In [48]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
)

def validate(model, loader, criterion):
    model.eval()

    total_loss = 0

    preds_all = []
    labels_all = []

    with torch.no_grad():
        for eeg, fnirs, labels in loader:
            eeg = eeg.to(DEVICE)
            fnirs = fnirs.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(eeg, fnirs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)

            preds_all.extend(
                preds.cpu().numpy()
            )

            labels_all.extend(
                labels.cpu().numpy()
            )

    acc = accuracy_score(
        labels_all,
        preds_all
    )

    bal_acc = balanced_accuracy_score(
        labels_all,
        preds_all
    )

    return (
        total_loss / len(loader),
        acc,
        bal_acc,
        preds_all,
        labels_all
    )

### Training Phase

In [49]:
NUM_CLASSES = 4
BATCH_SIZE = 32
LR = 1e-3
EPOCHS = 10
EMBED_DIM = 128
EEG_WINDOW_SIZE = 250
FNIRS_DELAY_START = 128
FNIRS_DELAY_END = 512

In [50]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import (
    LeaveOneGroupOut,
    GroupShuffleSplit
)
from sklearn.metrics import (
    confusion_matrix,
    classification_report
)


logo = LeaveOneGroupOut()

fold_scores = []

for fold, (
    train_idx,
    test_idx
) in enumerate(

    logo.split(
        X_eeg,
        y,
        groups=subject_ids
    )

):

    print("\n=================================")
    print(f"FOLD {fold+1}")
    print("=================================")

    # =====================================================
    # TEST SUBJECT
    # =====================================================

    test_subject = np.unique(
        subject_ids[test_idx]
    )

    print("TEST SUBJECT:", test_subject)

    # =====================================================
    # TRAIN + TEST SPLIT
    # =====================================================

    X_eeg_train_full = X_eeg[train_idx]
    X_fnirs_train_full = X_fnirs[train_idx]

    y_train_full = y[train_idx]

    train_subjects_full = subject_ids[
        train_idx
    ]

    X_eeg_test = X_eeg[test_idx]
    X_fnirs_test = X_fnirs[test_idx]

    y_test = y[test_idx]

    # =====================================================
    # INNER VALIDATION SPLIT
    # =====================================================

    gss = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    inner_train_idx, val_idx = next(

        gss.split(
            X_eeg_train_full,
            y_train_full,
            groups=train_subjects_full
        )

    )

    X_eeg_train = X_eeg_train_full[
        inner_train_idx
    ]

    X_fnirs_train = X_fnirs_train_full[
        inner_train_idx
    ]

    y_train = y_train_full[
        inner_train_idx
    ]

    X_eeg_val = X_eeg_train_full[
        val_idx
    ]

    X_fnirs_val = X_fnirs_train_full[
        val_idx
    ]

    y_val = y_train_full[
        val_idx
    ]

    # =====================================================
    # DATASET
    # =====================================================

    train_dataset = MultiModalDataset(
        X_eeg_train,
        X_fnirs_train,
        y_train
    )

    val_dataset = MultiModalDataset(
        X_eeg_val,
        X_fnirs_val,
        y_val
    )

    test_dataset = MultiModalDataset(
        X_eeg_test,
        X_fnirs_test,
        y_test
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # =====================================================
    # MODEL
    # =====================================================

    eeg_channels = X_eeg.shape[1]

    fnirs_channels = X_fnirs.shape[1]

    model = MultiModalNet(
        eeg_channels,
        fnirs_channels,
        NUM_CLASSES
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.1
    )

    scaler = torch.amp.GradScaler("cuda")

    best_val_acc = 0

    # =====================================================
    # TRAINING
    # =====================================================

    for epoch in range(EPOCHS):

        train_loss, train_acc, train_bal_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            scaler
        )

        val_loss, val_acc, val_bal_acc, _, _ = validate(
            model,
            val_loader,
            criterion
        )

        scheduler.step()

        if val_bal_acc > best_val_acc:

            best_val_acc = val_bal_acc

            torch.save(model.state_dict(), f"best_fold_{fold}.pth")

        print(
            f"Epoch {epoch+1:03d} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val Bal Acc: {val_bal_acc:.4f}"
        )

    # =====================================================
    # LOAD BEST MODEL
    # =====================================================

    model.load_state_dict(torch.load(f"best_fold_{fold}.pth",weights_only=True))

    # =====================================================
    # TEST
    # =====================================================

    test_loss, test_acc, test_bal_acc, preds, labels = validate(
        model,
        test_loader,
        criterion
    )

    print("\nTEST ACC:", test_acc)

    print("TEST BAL ACC:", test_bal_acc)

    cm = confusion_matrix(
        labels,
        preds
    )

    print("\nCONFUSION MATRIX")
    print(cm)

    print("\nCLASSIFICATION REPORT")

    print(
        classification_report(
            labels,
            preds
        )
    )

    fold_scores.append(
        test_bal_acc
    )

# =========================================================
# FINAL RESULT
# =========================================================

print("\n=================================")
print("FINAL LOSO RESULT")
print("=================================")

print(
    f"MEAN BALANCED ACC: "
    f"{np.mean(fold_scores):.4f}"
)

print(
    f"STD BALANCED ACC: "
    f"{np.std(fold_scores):.4f}"
)


FOLD 1
TEST SUBJECT: ['sub_01']
Epoch 001 | Train Acc: 0.2590 | Val Acc: 0.2857 | Val Bal Acc: 0.2856
Epoch 002 | Train Acc: 0.2897 | Val Acc: 0.2824 | Val Bal Acc: 0.2834
Epoch 003 | Train Acc: 0.2960 | Val Acc: 0.3021 | Val Bal Acc: 0.3030
Epoch 004 | Train Acc: 0.3098 | Val Acc: 0.3030 | Val Bal Acc: 0.3041
Epoch 005 | Train Acc: 0.3096 | Val Acc: 0.2874 | Val Bal Acc: 0.2874
Epoch 006 | Train Acc: 0.3231 | Val Acc: 0.2890 | Val Bal Acc: 0.2897
Epoch 007 | Train Acc: 0.3438 | Val Acc: 0.2865 | Val Bal Acc: 0.2874
Epoch 008 | Train Acc: 0.3632 | Val Acc: 0.2898 | Val Bal Acc: 0.2906
Epoch 009 | Train Acc: 0.3704 | Val Acc: 0.2865 | Val Bal Acc: 0.2871
Epoch 010 | Train Acc: 0.3711 | Val Acc: 0.2808 | Val Bal Acc: 0.2813

TEST ACC: 0.3492063492063492
TEST BAL ACC: 0.351038623823434

CONFUSION MATRIX
[[ 6 12 24 37]
 [ 6 16 43 15]
 [ 4  8 37 29]
 [ 5  9 13 51]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.29      0.08      0.12     

C:\Users\aldha\AppData\Local\Temp\ipykernel_27080\4239702088.py:222: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


Epoch 001 | Train Acc: 0.2679 | Val Acc: 0.3027 | Val Bal Acc: 0.3038
Epoch 002 | Train Acc: 0.2820 | Val Acc: 0.3076 | Val Bal Acc: 0.3103
Epoch 003 | Train Acc: 0.2920 | Val Acc: 0.3150 | Val Bal Acc: 0.3165
Epoch 004 | Train Acc: 0.3129 | Val Acc: 0.3142 | Val Bal Acc: 0.3152
Epoch 005 | Train Acc: 0.3190 | Val Acc: 0.2945 | Val Bal Acc: 0.2954
Epoch 006 | Train Acc: 0.3252 | Val Acc: 0.3060 | Val Bal Acc: 0.3073
Epoch 007 | Train Acc: 0.3369 | Val Acc: 0.2961 | Val Bal Acc: 0.2971
Epoch 008 | Train Acc: 0.3402 | Val Acc: 0.3060 | Val Bal Acc: 0.3069
Epoch 009 | Train Acc: 0.3594 | Val Acc: 0.3027 | Val Bal Acc: 0.3041
Epoch 010 | Train Acc: 0.3476 | Val Acc: 0.3084 | Val Bal Acc: 0.3097

TEST ACC: 0.3089171974522293
TEST BAL ACC: 0.30919344368711454

CONFUSION MATRIX
[[ 1 19 14 44]
 [ 0 17 38 24]
 [ 1 20 25 33]
 [ 0 15  9 54]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.50      0.01      0.03        78
           1       0.24  

C:\Users\aldha\AppData\Local\Temp\ipykernel_27080\4239702088.py:222: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


Epoch 001 | Train Acc: 0.2690 | Val Acc: 0.3166 | Val Bal Acc: 0.3169
Epoch 002 | Train Acc: 0.3055 | Val Acc: 0.2856 | Val Bal Acc: 0.2852
Epoch 003 | Train Acc: 0.3073 | Val Acc: 0.3126 | Val Bal Acc: 0.3123
Epoch 004 | Train Acc: 0.3183 | Val Acc: 0.3150 | Val Bal Acc: 0.3147
Epoch 005 | Train Acc: 0.3180 | Val Acc: 0.2991 | Val Bal Acc: 0.2988
Epoch 006 | Train Acc: 0.3346 | Val Acc: 0.3063 | Val Bal Acc: 0.3055
Epoch 007 | Train Acc: 0.3407 | Val Acc: 0.3039 | Val Bal Acc: 0.3035
Epoch 008 | Train Acc: 0.3507 | Val Acc: 0.3126 | Val Bal Acc: 0.3123
Epoch 009 | Train Acc: 0.3693 | Val Acc: 0.3166 | Val Bal Acc: 0.3160
Epoch 010 | Train Acc: 0.3402 | Val Acc: 0.3071 | Val Bal Acc: 0.3065

TEST ACC: 0.31521739130434784
TEST BAL ACC: 0.32165667625745953

CONFUSION MATRIX
[[ 9  9 22 35]
 [ 8 16 21 24]
 [ 5 12 25 22]
 [ 4  6 21 37]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.35      0.12      0.18        75
           1       0.37 

...

C:\Users\aldha\AppData\Local\Temp\ipykernel_27080\4239702088.py:222: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


Epoch 001 | Train Acc: 0.2826 | Val Acc: 0.3302 | Val Bal Acc: 0.3300
Epoch 002 | Train Acc: 0.2940 | Val Acc: 0.3246 | Val Bal Acc: 0.3252
Epoch 003 | Train Acc: 0.2834 | Val Acc: 0.3191 | Val Bal Acc: 0.3192
Epoch 004 | Train Acc: 0.2994 | Val Acc: 0.3302 | Val Bal Acc: 0.3307
Epoch 005 | Train Acc: 0.3148 | Val Acc: 0.3341 | Val Bal Acc: 0.3347
Epoch 006 | Train Acc: 0.3169 | Val Acc: 0.3175 | Val Bal Acc: 0.3178
Epoch 007 | Train Acc: 0.3496 | Val Acc: 0.3191 | Val Bal Acc: 0.3195
Epoch 008 | Train Acc: 0.3511 | Val Acc: 0.3286 | Val Bal Acc: 0.3290
Epoch 009 | Train Acc: 0.3523 | Val Acc: 0.3262 | Val Bal Acc: 0.3267
Epoch 010 | Train Acc: 0.3559 | Val Acc: 0.3191 | Val Bal Acc: 0.3195

TEST ACC: 0.22083333333333333
TEST BAL ACC: 0.21164190573770492

CONFUSION MATRIX
[[ 0  5 29 20]
 [ 6  7 29 19]
 [ 3 15 29 17]
 [ 7  7 30 17]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        54
           1       0.21 

C:\Users\aldha\AppData\Local\Temp\ipykernel_27080\4239702088.py:222: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


Epoch 001 | Train Acc: 0.2653 | Val Acc: 0.3436 | Val Bal Acc: 0.3436
Epoch 002 | Train Acc: 0.2930 | Val Acc: 0.3040 | Val Bal Acc: 0.3045
Epoch 003 | Train Acc: 0.2854 | Val Acc: 0.2969 | Val Bal Acc: 0.2975
Epoch 004 | Train Acc: 0.2935 | Val Acc: 0.3199 | Val Bal Acc: 0.3202
Epoch 005 | Train Acc: 0.3081 | Val Acc: 0.3222 | Val Bal Acc: 0.3226
Epoch 006 | Train Acc: 0.3180 | Val Acc: 0.3159 | Val Bal Acc: 0.3163
Epoch 007 | Train Acc: 0.3250 | Val Acc: 0.3175 | Val Bal Acc: 0.3179
Epoch 008 | Train Acc: 0.3343 | Val Acc: 0.3222 | Val Bal Acc: 0.3227
Epoch 009 | Train Acc: 0.3376 | Val Acc: 0.3310 | Val Bal Acc: 0.3313
Epoch 010 | Train Acc: 0.3558 | Val Acc: 0.3222 | Val Bal Acc: 0.3226

TEST ACC: 0.273972602739726
TEST BAL ACC: 0.2803504764711661

CONFUSION MATRIX
[[ 0 28  7 20]
 [ 0 26  8 20]
 [ 0 28  7 23]
 [ 0 20  5 27]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        55
           1       0.25    

C:\Users\aldha\AppData\Local\Temp\ipykernel_27080\4239702088.py:222: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(
C:\Users\aldha\Projects\eeg-fnirs-handwriting\

## Submission

In [53]:
import pandas as pd

df_test = get_dataset(type="test")
print(df_test.head())

      trial_id subject  session  onset_sec
0  sub_21_1_00  sub_21        1     24.495
1  sub_21_1_01  sub_21        1     49.347
2  sub_21_1_02  sub_21        1     74.205
3  sub_21_1_03  sub_21        1     99.075
4  sub_21_1_04  sub_21        1    123.933


In [54]:
import mne

mne.set_log_level("ERROR")
last_subject=""
last_session=0
raw=None
df_fnirs=None

EEG_TMIN = 0.0
EEG_TMAX = 4.0

# Dari explanatory data analysis
FNIRS_FS = 64

# 0.01 Hz ≤ f ≤ 0.2 Hz
# Kita hanya ambil perubahan sinyal yang:
# tidak terlalu lambat
# tidak terlalu cepat
FNIRS_LOW = 0.01
FNIRS_HIGH = 0.2

# Dimulai + 2 Second = menunggu respons aliran darah mulai muncul.
FNIRS_TMIN = 2.0
FNIRS_TMAX = 8.0

X_test_eeg = []
X_test_fnirs = []
trial_ids = []
subject_ids = []

for _, row in tqdm(
    df_test.iterrows(),
    total=len(df_test)
):
    subject = row["subject"]
    session = row["session"]
    onset_sec = row["onset_sec"]
    trial_id = row["trial_id"]

    if last_subject != subject or last_session != session:
        # EEG
        raw = read_bdf(subject, session)
        raw = preprocessing_bdf(raw)

        # fNIRS
        df_fnirs = read_fnirs(subject, session)
        df_fnirs = preprocessing_fnirs(df_fnirs, FNIRS_FS, FNIRS_LOW, FNIRS_HIGH)
        
        last_subject = subject
        last_session = session
        
    # Segmentation BDF
    segment_eeg, label_eeg = segmentation_eeg(raw, onset_sec, label, EEG_TMIN, EEG_TMAX)
    
    # Segmentation fNIRS
    segment_fnirs, label_fnirs = segmentation_fnirs(
        df_fnirs,
        onset_sec,
        label,
        FNIRS_TMIN,
        FNIRS_TMAX,
        FNIRS_FS
    )
    if label_eeg == -1 or label_fnirs == -1:
        continue

    X_test_eeg.append(
        segment_eeg
    )

    X_test_fnirs.append(
        segment_fnirs
    )

    trial_ids.append(
        trial_id
    )
    subject_ids.append(subject)

X_test_eeg = np.array(X_test_eeg)
X_test_fnirs = np.array(X_test_fnirs)
subject_ids = np.array(subject_ids)
X_test_eeg = np.nan_to_num(
    X_test_eeg,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X_test_fnirs = np.nan_to_num(
    X_test_fnirs,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

# Normalize EEG
X_test_eeg = normalize_channel(X_test_eeg)

# Normalize fNIRS
X_test_fnirs = normalize_fnirs(X_test_fnirs)

print("Test EEG shape:")
print(X_test_eeg.shape)

print("Test fNIRS shape:")
print(X_test_fnirs.shape)


100%|██████████████████████████████████████████████████████████████████████████████| 3238/3238 [03:20<00:00, 16.18it/s]


Test EEG shape:
(3205, 27, 4000)
Test fNIRS shape:
(3205, 8, 384)


In [55]:
# =========================================================
# TEST DATASET
# =========================================================

class TestDataset(Dataset):
    def __init__(self, eeg, fnirs):
        self.eeg = eeg
        self.fnirs = fnirs

    def __len__(self):
        return len(self.eeg)

    def __getitem__(self, idx):
        eeg = torch.tensor(self.eeg[idx], dtype=torch.float32)
        fnirs = torch.tensor(self.fnirs[idx], dtype=torch.float32)

        return eeg, fnirs


test_dataset = TestDataset(X_test_eeg, X_test_fnirs)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# =========================================================
# LOAD BEST MODEL
# =========================================================

eeg_channels = X_test_eeg.shape[1]
fnirs_channels = X_test_fnirs.shape[1]
model = MultiModalNet(
    eeg_channels,
    fnirs_channels,
    NUM_CLASSES
).to(DEVICE)

# pakai fold terbaikmu
BEST_FOLD = 0

model.load_state_dict(
    torch.load(f"best_fold_{BEST_FOLD}.pth", weights_only=True)
)

model.eval()


# =========================================================
# INFERENCE
# =========================================================

predictions = []

with torch.no_grad():
    for eeg, fnirs in tqdm(test_loader):
        eeg = eeg.to(DEVICE)
        fnirs = fnirs.to(DEVICE)
        
        with torch.amp.autocast("cuda"):
            outputs = model(eeg, fnirs)
            
        preds = torch.argmax(outputs, dim=1)
        predictions.extend(preds.cpu().numpy())


predictions = np.array(predictions)

print("Predictions shape:")
print(predictions.shape)


# =========================================================
# CREATE SUBMISSION
# =========================================================

submission = pd.DataFrame({
    "trial_id": trial_ids,
    "label": predictions
})

print(submission.head())


# =========================================================
# SAVE CSV
# =========================================================

submission.to_csv(
    "submission.csv",
    index=False
)

print("\nsubmission.csv saved!")

100%|███████████████████████████████████████████████████████████████████████████████| 101/101 [00:00<00:00, 109.88it/s]


Predictions shape:
(3205,)
      trial_id  label
0  sub_21_1_00      2
1  sub_21_1_01      3
2  sub_21_1_02      3
3  sub_21_1_03      3
4  sub_21_1_04      2

submission.csv saved!
